# Preparación de datos electorales de Sonora

Esta libreta materializa los artefactos que consumen las libretas de análisis
(`01`–`04`). Todo queda en `data/processed/`:

| Artefacto | Contenido |
|---|---|
| `secciones_{año}.gpkg` | Secciones electorales de Sonora 2019–2025 (EPSG:4326) |
| `distritos_{año}.gpkg` | Distritos locales (21) disueltos por año |
| `municipios_sonora.gpkg` | Municipios (72) disueltos desde secciones 2024 (`ln_2024`/`pad_2024`) |
| `seccion_distrito_por_anio.csv` | Tabla `sección → distrito_l` por año |
| `conteos_sonora.csv.gz` | Conteos censales electorales 2009–2024 (sección×sexo×edad) |
| `censo_inegi_municipios.csv` | Totales municipales ITER 2010 y 2020 |
| `conapo_sonora.csv` | Población municipal por sexo y grupo quinquenal, 1990–2040 |
| `mun_ine_inegi.csv` | Traducción clave municipal INE ↔ `mpiocve` INEGI |

**Fuentes**: cartografía del INE (`data/raw/Mapas`), conteos censales del INE
(`data/Censos`), ITER del INEGI y proyecciones municipales de CONAPO
(datos.gob.mx).

In [1]:
import os
import unicodedata
from zipfile import ZipFile

import geopandas as gpd
import pandas as pd
import requests

from vothmo.config import FOLDER_DATA, FOLDER_RAW
from vothmo.repair_shp import repair_file

ENT_SONORA = 26
MAP_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
CONTEO_YEARS = [2009, 2012, 2015, 2018, 2021, 2024]

FOLDER_MAP = FOLDER_RAW / "Mapas"
DATA_MAPS = FOLDER_DATA / "Mapas"
PROCESSED = FOLDER_DATA / "processed"
RAW_INEGI = FOLDER_RAW / "inegi"
RAW_CONAPO = FOLDER_RAW / "conapo"
for d in (DATA_MAPS, PROCESSED, RAW_INEGI, RAW_CONAPO):
    d.mkdir(parents=True, exist_ok=True)

## 1. Cartografía electoral del INE

Extraemos los zips de `data/raw/Mapas/{año}` a `data/Mapas/{año}`. Cada capa
`SECCION` trae, por sección: `distrito_f`, `distrito_l`, `municipio` (clave
geoestadística), `tipo`, `ln` (lista nominal) y `pad` (padrón electoral).
Las columnas varían por año: 2019 usa `padron`/`lista`, 2020+ usa `pad`/`ln`,
2023 agrega `nombre` y 2025 viene en `SECCIONES_26/SECCION_26.*`.

In [2]:
for year_dir in sorted(FOLDER_MAP.iterdir()):
    if not year_dir.is_dir():
        continue
    dest = DATA_MAPS / year_dir.name
    for zp in sorted(year_dir.glob("*.zip")):
        marker = dest / zp.stem
        # Re-extrae solo si no hay ya un shapefile de esta capa
        ya = list(dest.rglob(zp.stem + "*.shp")) or list(dest.rglob(zp.stem.split('_')[0] + "*.shp"))
        if ya:
            continue
        dest.mkdir(exist_ok=True)
        with ZipFile(zp) as z:
            z.extractall(dest)

for year in MAP_YEARS:
    shps = sorted((DATA_MAPS / str(year)).rglob("*.shp"))
    print(year, [p.relative_to(DATA_MAPS / str(year)).as_posix() for p in shps])

# Los shapefiles del INE 2019/2020 traen anillos sin cerrar (ver
# vothmo/repair_shp.py). Se reparan in-situ una sola vez.
for shp in sorted(DATA_MAPS.rglob("*.shp")):
    n, info = repair_file(shp)
    if n:
        print(f"reparado {shp.relative_to(DATA_MAPS)}: {n} anillos")

2019 ['DISTRITO_FEDERAL.shp', 'ENTIDAD.shp', 'LOCALIDAD.shp', 'LOCALIDADES_RURALES.shp', 'MANZANAS.shp', 'MUNICIPIO.shp', 'SECCION.shp']
2020 ['DISTRITO_FEDERAL.shp', 'ENTIDAD.shp', 'LOCALIDAD.shp', 'MANZANAS.shp', 'MUNICIPIO.shp', 'SECCION.shp']
2021 ['DISTRITO_FEDERAL.shp', 'ENTIDAD.shp', 'LOCALIDAD.shp', 'MANZANAS.shp', 'MUNICIPIO.shp', 'SECCION.shp']
2022 ['DISTRITO_FEDERAL/DISTRITO_FEDERAL.shp', 'ENTIDAD/ENTIDAD.shp', 'LOCALIDAD/LOCALIDAD.shp', 'MANZANAS/MANZANA.shp', 'MUNICIPIO/MUNICIPIO.shp', 'SECCION/SECCION.shp']
2023 ['DISTRITO_FEDERAL.shp', 'ENTIDAD.shp', 'Localidad.shp', 'MUNICIPIO.shp', 'Manzanas.shp', 'SECCION.shp']
2024 ['DISTRITO_FEDERAL/DISTRITO_FEDERAL.shp', 'ENTIDAD/ENTIDAD.shp', 'LOCALIDADES/LOCALIDADES.shp', 'MANZANAS/MANZ4.shp', 'MUNICIPIO/MUNICIPIO.shp', 'SECCION/SECCION.shp']
2025 ['DISTRITO_FEDERAL_26/DISTRITO_FEDERAL_26.shp', 'ENTIDAD_26/ENTIDAD_26.shp', 'LOCALIDAD_26/LOCALIDAD_26.shp', 'MANZANAS_26/MANZANAS_2026.shp', 'MUNICIPIO_26/MUNICIPIO_26.shp', 'SECCION

In [3]:
def cargar_secciones(year: int) -> gpd.GeoDataFrame:
    """Lee la capa SECCION del año y devuelve solo Sonora, normalizada."""
    shp = sorted((DATA_MAPS / str(year)).rglob("SECCION*.shp"))[0]
    gdf = gpd.read_file(shp)
    gdf.columns = [c.lower() for c in gdf.columns]
    gdf = gdf.rename(columns={"padron": "pad", "lista": "ln"})
    gdf = gdf[gdf["entidad"] == ENT_SONORA].copy()
    keep = ["entidad", "distrito_f", "distrito_l", "municipio", "seccion",
            "tipo", "ln", "pad", "geometry"]
    return gdf[[c for c in keep if c in gdf.columns]].to_crs(4326)


secciones = {}
for year in MAP_YEARS:
    g = cargar_secciones(year)
    assert g["distrito_l"].nunique() == 21, year
    secciones[year] = g
    g.to_file(PROCESSED / f"secciones_{year}.gpkg", driver="GPKG")
    print(year, "|", len(g), "secciones | pad total:", int(g["pad"].sum()))

2019 | 1533 secciones | pad total: 2032271


2020 | 1533 secciones | pad total: 2108953


2021 | 1528 secciones | pad total: 2191210


2022 | 1528 secciones | pad total: 2140362


2023 | 1535 secciones | pad total: 2221477


2024 | 1573 secciones | pad total: 2295226


2025 | 1667 secciones | pad total: 2317487


In [4]:
# Capas disueltas: distritos locales por año y municipios (año de referencia)
for year, g in secciones.items():
    d = g.dissolve(by="distrito_l", aggfunc="sum").reset_index()
    d = d[["distrito_l", "ln", "pad", "geometry"]]
    d.to_file(PROCESSED / f"distritos_{year}.gpkg", driver="GPKG")

municipios = (secciones[2024]
              .dissolve(by="municipio", aggfunc="sum")
              .reset_index()
              .rename(columns={"ln": "ln_2024", "pad": "pad_2024"})
              [["municipio", "ln_2024", "pad_2024", "geometry"]])
assert municipios["municipio"].nunique() == 72
municipios.to_file(PROCESSED / "municipios_sonora.gpkg", driver="GPKG")
municipios.head()

,municipio,ln_2024,pad_2024,geometry
0,1,2635,2642,"POLYGON ((-110.18536 29.86637, -110.18648 29.8..."
1,2,71928,72031,"POLYGON ((-109.40243 31.06561, -109.40332 31.0..."
2,3,7121,7138,"POLYGON ((-111.79524 30.70909, -111.79489 30.7..."
3,4,1131,1135,"POLYGON ((-108.85504 28.88398, -108.85708 28.8..."
4,5,2970,2982,"POLYGON ((-110.45914 30.18036, -110.45931 30.1..."


In [5]:
# Tabla de correspondencia sección -> distrito por año (insumo de 01-04)
piv = pd.concat(
    g[["seccion", "municipio", "distrito_l", "distrito_f"]].assign(anio=year)
    for year, g in secciones.items()
)
piv.to_csv(PROCESSED / "seccion_distrito_por_anio.csv", index=False)
piv.groupby("anio").size()

anio
2019    1533
2020    1533
2021    1528
2022    1528
2023    1535
2024    1573
2025    1667
dtype: int64

## 2. Conteos censales electorales (INE, 2009–2024)

Concatenamos los seis archivos de Sonora. La columna `DEL` (distrito electoral
local) viene vacía en 1 223 filas de 2024; se imputa uniendo por `seccion` con
la cartografía del mismo año (verificado: resuelve todas y no contradice `DEL`).
`SEXO`: 0 hombres, 1 mujeres, 2 no binario (solo 2024). Variables derivadas:
`lns = ln - ns` (registros válidos) y `pc = sv / lns` (participación).

In [6]:
frames = []
for y in CONTEO_YEARS:
    p = (FOLDER_DATA / "Censos"
         / f"DatosAbiertos_DECEyEC_ConteosCensales{y}"
         / f"DatosAbiertos_DECEyEC_ConteosCensales{y}"
         / f"datosabiertos_deceyec_conteoscensales{y}_son.csv")
    df = pd.read_csv(p, encoding="utf-8-sig", low_memory=False)
    df.columns = [c.lower() for c in df.columns]
    frames.append(df)

conteos = pd.concat(frames, ignore_index=True)
print(conteos.shape)
conteos.head()

(1181180, 16)


,aelec,feleccion,edocve,edonom,mpiocve,mpionom,seccion,tiposec,def,del,sexo,edad,ln,sv,nv,ns
0,2009,5/7/2009,26,SONORA,1,Aconchi,1,M,4,18,0,18,21,14,7,0
1,2009,5/7/2009,26,SONORA,1,Aconchi,1,M,4,18,0,19,16,15,1,0
2,2009,5/7/2009,26,SONORA,1,Aconchi,1,M,4,18,0,20,10,8,2,0
3,2009,5/7/2009,26,SONORA,1,Aconchi,1,M,4,18,0,21,11,9,2,0
4,2009,5/7/2009,26,SONORA,1,Aconchi,1,M,4,18,0,22,17,15,2,0


In [7]:
# DEL puede venir como texto con espacios; homogeneizar a numérico antes.
# `def` (distrito federal) tambien llega con tipos mixtos
conteos["def"] = pd.to_numeric(conteos["def"], errors="coerce").astype("Int64")
conteos["del"] = pd.to_numeric(conteos["del"], errors="coerce")

# Imputar DEL faltante con la cartografía del año correspondiente (o el mapa
# más cercano). Solo 2024 tiene faltantes.
def mapa_distrito(anio_conteo: int) -> pd.Series:
    year = min(MAP_YEARS, key=lambda y: abs(y - anio_conteo))
    return secciones[year].set_index("seccion")["distrito_l"]


for y in CONTEO_YEARS:
    mask = (conteos["aelec"] == y) & conteos["del"].isna()
    if mask.any():
        conteos.loc[mask, "del"] = conteos.loc[mask, "seccion"].map(mapa_distrito(y))

print("DEL faltantes tras imputar:", int(conteos["del"].isna().sum()))

conteos["del"] = conteos["del"].astype("Int64")
conteos["lns"] = conteos["ln"] - conteos["ns"]
conteos["pc"] = conteos["sv"] / conteos["lns"]
conteos.to_csv(PROCESSED / "conteos_sonora.csv.gz", index=False,
               compression="gzip")
conteos.groupby("aelec")["ln"].sum()

DEL faltantes tras imputar: 2769


aelec
2009    1826173
2012    1718637
2015    1967629
2018    2101726
2021    2187666
2024    2291013
Name: ln, dtype: int64

## 3. Población: INEGI y CONAPO

Se descargan (si no existen ya en `data/raw/`):

- **ITER Censo 2020** (Sonora) — `inegi.org.mx`
- **ITER Censo 2010** (nacional, contiene el zip por entidad) — `inegi.org.mx`
- **CONAPO** población a mitad de año por municipio, sexo y grupos quinquenales
  de edad 1990–2040 — `datos.gob.mx` (requiere User-Agent de navegador)

Si una descarga falla, coloca el archivo manualmente en `data/raw/inegi/` o
`data/raw/conapo/` con el mismo nombre y vuelve a ejecutar.

> Nota: las *Estadísticas Censales a Escalas Geoelectorales* (INEGI, por
> sección) no están disponibles en descarga directa; el análisis
> padrón-vs-censo a nivel municipal usa CONAPO/ITER.

In [8]:
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:128.0) Gecko/20100101 Firefox/128.0"}

DESCARGAS = {
    RAW_INEGI / "iter_26_cpv2020_csv.zip":
        "https://www.inegi.org.mx/contenidos/programas/ccpv/2020/"
        "datosabiertos/iter/iter_26_cpv2020_csv.zip",
    RAW_INEGI / "iter_2010.zip":
        "https://www.inegi.org.mx/contenidos/programas/ccpv/2010/"
        "datosabiertos/iter_2010.zip",
    RAW_CONAPO / "pobproy_quinq1.csv":
        "https://www.datos.gob.mx/dataset/"
        "f2b9b220-3ef7-4e3a-bde6-87e1dac78c6a/resource/"
        "3c3092be-583e-4490-8c23-67ef9a64b198/download/pobproy_quinq1.csv",
}

for dest, url in DESCARGAS.items():
    if dest.exists() and dest.stat().st_size > 0:
        print("ya existe:", dest.name)
        continue
    resp = requests.get(url, headers=UA, timeout=600)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print("descargado:", dest.name, f"({dest.stat().st_size / 1e6:.1f} MB)")

ya existe: iter_26_cpv2020_csv.zip
ya existe: iter_2010.zip
ya existe: pobproy_quinq1.csv


In [9]:
# ITER 2020 (Sonora) + ITER 2010 (zip nacional -> zip interno de Sonora)
iter_dir = PROCESSED / "iter"
iter_dir.mkdir(exist_ok=True)

with ZipFile(RAW_INEGI / "iter_26_cpv2020_csv.zip") as z:
    with z.open("iter_26_cpv2020/conjunto_de_datos/conjunto_de_datos_iter_26CSV20.csv") as f:
        iter2020 = pd.read_csv(f, encoding="utf-8-sig", low_memory=False)
iter2020.columns = [c.lower() for c in iter2020.columns]

with ZipFile(RAW_INEGI / "iter_2010.zip") as z:
    with z.open("iter_26_2010_csv.zip") as zin:
        with ZipFile(zin) as z2:
            with z2.open("conjunto_de_datos/iter_26_cpv2010.csv") as f:
                iter2010 = pd.read_csv(f, encoding="utf-8-sig", low_memory=False)
iter2010.columns = [c.lower() for c in iter2010.columns]


def totales_municipales(df: pd.DataFrame, anio: int) -> pd.DataFrame:
    tot = df[df["nom_loc"].str.contains("Total del Municipio", case=False,
                                       na=False)].copy()
    tot["mpiocve"] = pd.to_numeric(tot["mun"], errors="coerce").astype("Int64")
    tot["anio"] = anio
    cols = ["anio", "mpiocve", "nom_mun", "pobtot", "pobmas", "pobfem",
            "p_18ymas", "p_18ymas_m", "p_18ymas_f", "p_15a17", "p_18a24",
            "p_60ymas"]
    cols = [c for c in cols if c in tot.columns]
    for c in cols[3:]:
        tot[c] = pd.to_numeric(tot[c], errors="coerce")
    return tot[cols]


censo_inegi = pd.concat([totales_municipales(iter2010, 2010),
                         totales_municipales(iter2020, 2020)],
                        ignore_index=True)
assert censo_inegi["mpiocve"].nunique() == 72
censo_inegi.to_csv(PROCESSED / "censo_inegi_municipios.csv", index=False)
censo_inegi.head(10)

,anio,mpiocve,nom_mun,pobtot,pobmas,pobfem,p_18ymas,p_18ymas_m,p_18ymas_f,p_15a17,p_18a24,p_60ymas
0,2010,1,Aconchi,2637,1405,1232,1755,913,842,141,279,367
1,2010,2,Agua Prieta,79138,40117,39021,48002,24179,23823,4802,9888,4759
2,2010,3,Alamos,25848,13497,12351,16851,8798,8053,1549,2865,3587
3,2010,4,Altar,9049,4624,4425,5494,2813,2681,485,1076,724
4,2010,5,Arivechi,1253,650,603,853,450,403,83,114,235
5,2010,6,Arizpe,3037,1571,1466,2100,1093,1007,168,283,544
6,2010,7,Atil,625,326,299,431,220,211,28,61,118
7,2010,8,Bacadéhuachi,1252,667,585,858,465,393,49,112,207
8,2010,9,Bacanora,784,417,367,574,309,265,32,60,198
9,2010,10,Bacerac,1467,761,706,947,502,445,98,160,189


In [10]:
# CONAPO: filtrar Sonora y estimar población 18+.
# 18+ = total - (0-4) - (5-9) - (10-14) - 0.6 * (15-19)  [edades 15-17]
conapo = pd.read_csv(RAW_CONAPO / "pobproy_quinq1.csv")
conapo.columns = [c.lower() for c in conapo.columns]
conapo = conapo.rename(columns={"ano": "anio"})
conapo = conapo[conapo["clave_ent"] == ENT_SONORA].copy()
conapo["mpiocve"] = (conapo["clave"] % 1000).astype("Int64")
conapo["sexo"] = conapo["sexo"].map({"HOMBRES": 0, "MUJERES": 1})

menores = ["pob_00_04", "pob_05_09", "pob_010_014"]
conapo["pob_18mas"] = (conapo["pob_total"] - conapo[menores].sum(axis=1)
                       - 0.6 * conapo["pob_015_019"])

cols = ["anio", "mpiocve", "nom_mun", "sexo", "pob_total", "pob_18mas",
        "pob_015_019", "pob_20_24", "pob_25_29", "pob_30_34", "pob_35_39",
        "pob_40_44", "pob_45_49", "pob_50_54", "pob_55_59", "pob_60_64",
        "pob_65_69", "pob_70_74", "pob_75_79", "pob_80_84", "pob_85_mm"]
conapo = conapo[cols]
assert conapo["mpiocve"].nunique() == 72
conapo.to_csv(PROCESSED / "conapo_sonora.csv", index=False)
conapo.head()

,anio,mpiocve,nom_mun,sexo,pob_total,pob_18mas,pob_015_019,pob_20_24,pob_25_29,pob_30_34,...,pob_40_44,pob_45_49,pob_50_54,pob_55_59,pob_60_64,pob_65_69,pob_70_74,pob_75_79,pob_80_84,pob_85_mm
194514,1990,1,Aconchi,0,1270,721.8,142,104,83,70,...,64,64,56,44,36,31,23,16,8,2
194515,1990,1,Aconchi,1,1140,650.2,123,93,80,73,...,55,47,42,39,33,25,22,16,6,4
194516,1991,1,Aconchi,0,1254,724.6,139,106,84,70,...,62,62,55,45,38,32,24,16,8,2
194517,1991,1,Aconchi,1,1138,658.0,125,94,80,74,...,55,48,44,40,34,26,21,15,7,4
194518,1992,1,Aconchi,0,1238,729.6,134,108,85,71,...,61,60,55,46,39,33,25,16,9,3


### Mapeo municipal INE ↔ INEGI

La cartografía y los conteos del INE numeran los municipios con la
**clave municipal del INE**, que **no coincide** con el catálogo
`mpiocve` del INEGI (el que usan ITER y CONAPO). Se construye la
traducción por nombre normalizado y se guarda como
`mun_ine_inegi.csv`; además se enriquecen las capas de secciones y
municipios con `mpiocve` + `nom_mun` para que las libretas de análisis
crucen directamente contra las fuentes poblacionales.

In [11]:
def _norm_nom(s: pd.Series) -> pd.Series:
    """Mayúsculas, sin acentos ni paréntesis; GRAL -> GENERAL."""
    return (s.str.upper().str.strip()
             .map(lambda x: "".join(c for c in unicodedata.normalize("NFKD", x)
                                    if not unicodedata.combining(c)))
             .str.replace(r"\([^)]*\)", "", regex=True)
             .str.replace(r"\bGRAL\.?\b", "GENERAL", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())


ine_mun = (conteos[["mpiocve", "mpionom"]]
           .groupby("mpiocve").mpionom.agg(lambda x: x.mode().iloc[0])
           .reset_index())
munmap = (ine_mun.assign(key=_norm_nom(ine_mun.mpionom))
          .merge(conapo.assign(key=_norm_nom(conapo.nom_mun))
                 [["mpiocve", "nom_mun", "key"]].drop_duplicates("key"),
                 on="key", suffixes=("_ine", ""))
          .rename(columns={"mpiocve_ine": "mun_ine"})
          [["mun_ine", "mpiocve", "mpionom", "nom_mun"]])
assert munmap.mpiocve.notna().all() and len(munmap) == 72
munmap.to_csv(PROCESSED / "mun_ine_inegi.csv", index=False)

# Enriquecer secciones (en memoria y en disco) con mpiocve + nom_mun
_add = munmap[["mun_ine", "mpiocve", "nom_mun"]]
for year in MAP_YEARS:
    secciones[year] = (secciones[year]
                       .merge(_add, left_on="municipio", right_on="mun_ine")
                       .drop(columns="mun_ine"))
    secciones[year].to_file(PROCESSED / f"secciones_{year}.gpkg",
                            driver="GPKG")

municipios = (municipios.merge(_add, left_on="municipio", right_on="mun_ine")
              .drop(columns="mun_ine"))
municipios.to_file(PROCESSED / "municipios_sonora.gpkg", driver="GPKG")

piv = (piv.merge(munmap[["mun_ine", "mpiocve"]],
                left_on="municipio", right_on="mun_ine")
       .drop(columns="mun_ine"))
piv.to_csv(PROCESSED / "seccion_distrito_por_anio.csv", index=False)

munmap.head()

,mun_ine,mpiocve,mpionom,nom_mun
0,1,1,Aconchi,Aconchi
1,2,2,Agua Prieta,Agua Prieta
2,3,4,Altar,Altar
3,4,5,Arivechi,Arivechi
4,5,6,Arizpe,Arizpe


### Capas simplificadas para visualización

Las geometrías del INE son muy detalladas (>100 MB por capa nacional).
Para las gráficas de mapa se generan versiones simplificadas
(`simplify` + `set_precision`) con el sufijo `_simp`:
`secciones_simp_{año}.gpkg`, `distritos_simp_{año}.gpkg`,
`municipios_simp.gpkg`. Úsalas siempre en figuras; conserva las capas
completas para cálculos espaciales exactos.

In [12]:
def simplificar(g: gpd.GeoDataFrame, tolerancia: float) -> gpd.GeoDataFrame:
    s = g.copy()
    s["geometry"] = (s.geometry
                     .simplify(tolerancia, preserve_topology=True)
                     .set_precision(0.0001))
    return s


for year, g in secciones.items():
    simplificar(g, 0.0008).to_file(
        PROCESSED / f"secciones_simp_{year}.gpkg", driver="GPKG")
    simplificar(distritos_gdf := g.dissolve(by="distrito_l", aggfunc="sum")
                .reset_index()[["distrito_l", "ln", "pad", "geometry"]],
                0.002).to_file(
        PROCESSED / f"distritos_simp_{year}.gpkg", driver="GPKG")

simplificar(municipios, 0.002).to_file(
    PROCESSED / "municipios_simp.gpkg", driver="GPKG")
print("capas simplificadas escritas")

capas simplificadas escritas


## Resumen

Artefactos generados en `data/processed/`:

- `secciones_{2019..2025}.gpkg`, `distritos_{2019..2025}.gpkg`,
  `municipios_sonora.gpkg`
- `seccion_distrito_por_anio.csv` — correspondencia sección→distrito por año
- `conteos_sonora.csv.gz` — conteos electorales 2009–2024 con `del` imputado,
  `lns` y `pc`
- `censo_inegi_municipios.csv` — totales municipales 2010/2020
- `conapo_sonora.csv` — población municipal Sonora por sexo/edad, 1990–2040
- `mun_ine_inegi.csv` — clave municipal INE ↔ INEGI (las capas
  `secciones_*`/`municipios_*` ya traen `mpiocve` y `nom_mun`)

Siguientes libretas: `01_distritos_historia`, `02_demografia_electoral`,
`03_escenarios`, `04_padron_vs_censo`.